# Run03 (Final)

In [7]:
VERSION = 3

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor, Pool


from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge, LassoCV, ElasticNetCV
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, GroupShuffleSplit, GroupKFold
from scipy.stats import pearsonr, spearmanr


In [8]:
df = pd.read_csv(r"C:\Users\Pratham Shah\Downloads\Tubingen\Codes\MiCRM_Final_Dataset\community_index_v10.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nTarget (community_suppression) statistics:")
print(df['community_suppression'].describe().round(4))

print("\nRows per diversity level:")
print(df['diversity_level'].value_counts().sort_index())

print("\nRows per community size:")
print(df['community_size'].value_counts().sort_index())

Dataset shape: 75,000 rows × 168 columns

Target (community_suppression) statistics:
count    75000.0000
mean         0.5166
std          0.3328
min          0.0000
25%          0.2331
50%          0.4961
75%          0.8426
max          0.9995
Name: community_suppression, dtype: float64

Rows per diversity level:
diversity_level
1     15000
2     15000
4     15000
8     15000
16    15000
Name: count, dtype: int64

Rows per community size:
community_size
12    15000
16    15000
24    15000
32    15000
48    15000
Name: count, dtype: int64


In [9]:
# feature selection controlled by version

r0_cols     = [f"R0_c{k}"     for k in range(1, 21)]
active_cols = [f"active_c{k}" for k in range(1, 21)]

if VERSION == 1:
    species_cols = [f"Species_{i:04d}" for i in range(2, 102)]
    feature_cols = species_cols + r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Full (species + nutrients + size)"

elif VERSION == 2:
    feature_cols = r0_cols + active_cols + ['community_size', 'diversity_level']
    version_label = "Nutrients + size (no species)"

elif VERSION == 3:
    feature_cols = r0_cols + active_cols
    version_label = "Pure nutrients only (no species, no size)"

else:
    raise ValueError("VERSION must be 1, 2, or 3")

print(f"\nVersion: {version_label}")
print(f"Features: {len(feature_cols)}")

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise KeyError(f"{len(missing)} columns not found. First 3: {missing[:3]}\n"
                   f"Make sure you are loading community_index_v8_backup.csv (30,000 rows), "
                   f"not community_dataset_v8_backup.csv (792,000 rows).")

X = df[feature_cols]
y = df['community_suppression']
print(f"X: {X.shape}   y range: [{y.min():.3f}, {y.max():.3f}]")


Version: Pure nutrients only (no species, no size)
Features: 40
X: (75000, 40)   y range: [0.000, 0.999]


In [28]:
# train test split

strat_key = df["community_size"].astype(str) + "_" + df["diversity_level"].astype(str)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=strat_key
)
print(f"\nTrain: {X_train.shape[0]:,}   Test: {X_test.shape[0]:,}")


groups = df["broth_id"]  # Kept as Series
groups_train = df.loc[X_train.index, "broth_id"]  # Kept as Series



def inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42):
    gss = GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
    fit_idx, val_idx = next(gss.split(X_tr, y_tr, groups=groups_tr))
    return (
        X_tr.iloc[fit_idx],
        y_tr.iloc[fit_idx],
        X_tr.iloc[val_idx],
        y_tr.iloc[val_idx],
    )


Train: 60,000   Test: 15,000


In [11]:
# catboost initial fit

# note about l2_leaf_reg:
#  L2 regularisation on leaf values. Equivalent to
#  reg_lambda in XGBoost. Default is 3.0 (stronger than
#  XGBoost's default of 1.0)

model_init = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3.0,
    subsample=0.8,
    colsample_bylevel=0.8,
    early_stopping_rounds=30,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0,
    train_dir='catboost_info_v3'
)


Xi_fit, yi_fit, Xi_val, yi_val = inner_holdout(X_train, y_train, groups_train)
fit_pool = Pool(Xi_fit, label=yi_fit, cat_features=[])
val_pool = Pool(Xi_val, label=yi_val, cat_features=[])

model_init.fit(fit_pool, eval_set=val_pool)

yp_init = model_init.predict(X_test)
r2_init  = r2_score(y_test, yp_init)
mae_init = mean_absolute_error(y_test, yp_init)
print(f"CatBoost initial:  R²={r2_init:.4f}  MAE={mae_init:.4f}  "
      f"best_iter={model_init.best_iteration_}")

CatBoost initial:  R²=0.6713  MAE=0.1395  best_iter=264


In [12]:
# hyperparameter tuning

param_grid = {                              
    'depth':            [4,6,8],
    # depth is CatBoost's primary complexity control.
    # depth=4 → 16 leaves, depth=8 → 256 leaves.

    'learning_rate':    [0.01,0.05,0.1],

    'l2_leaf_reg':      [1.0,3.0,5.0,10.0],
    # CatBoost-specific. Higher = more regularisation.
    # Default 3.0 is already stronger than XGBoost/LightGBM defaults.
    # Try 10.0 for VERSION 3 (pure nutrient) where overfitting is more likely.

    'colsample_bylevel':[0.6,0.8,1.0],
}


group_cv = GroupKFold(n_splits=3)

base_est = CatBoostRegressor(
    iterations=300,
    subsample=0.8,
    early_stopping_rounds=None,   # disabled inside GridSearchCV
    use_best_model=False,         # disabled inside GridSearchCV
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=0,
    train_dir='catboost_info_v3'
)

grid_search = GridSearchCV(
    estimator=base_est, 
    param_grid=param_grid, 
    cv=group_cv,
    scoring='neg_mean_squared_error', 
    n_jobs=1, 
    verbose=1, 
    refit=True)

grid_search.fit(X_train, y_train, groups=groups_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV RMSE: {np.sqrt(-grid_search.best_score_):.4f}")

Fitting 3 folds for each of 108 candidates, totalling 324 fits

Best params: {'colsample_bylevel': 0.8, 'depth': 6, 'l2_leaf_reg': 3.0, 'learning_rate': 0.1}
Best CV RMSE: 0.2727


In [25]:
# final model

model_final = CatBoostRegressor(
    **grid_search.best_params_,
    iterations=1000,
    subsample=0.8,
    early_stopping_rounds=40,
    use_best_model=True,
    eval_metric='RMSE',
    random_seed=42,
    thread_count=-1,
    verbose=50,
    train_dir='catboost_info_v3'
)

Xf_fit, yf_fit, Xf_val, yf_val = inner_holdout(X_train, y_train, groups_train)
fit_pool = Pool(Xf_fit, label=yf_fit, cat_features=[])
val_pool = Pool(Xf_val, label=yf_val, cat_features=[])

model_final.fit(fit_pool, eval_set=val_pool)

evals    = model_final.get_evals_result()
tr_rmse  = evals['learn']['RMSE']
val_rmse = evals['validation']['RMSE']
rounds   = range(1, len(tr_rmse) + 1)
best_it  = model_final.best_iteration_

fig_tc, ax_tc = plt.subplots(figsize=(9, 4))
ax_tc.plot(rounds, tr_rmse,  color='steelblue',  lw=1.5, label='Train RMSE')
ax_tc.plot(rounds, val_rmse, color='darkorange', lw=1.5, label='Validation RMSE')
ax_tc.axvline(best_it + 1, color='red', linestyle='--', lw=1.2,
              label=f'Best iteration = {best_it + 1}')
ax_tc.set_xlabel('Boosting round')
ax_tc.set_ylabel('RMSE')
ax_tc.set_title(f'CatBoost Training Curve  --  best iter={best_it + 1},  val RMSE={val_rmse[best_it]:.4f}')
ax_tc.legend(fontsize=9)
ax_tc.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('catboost_training_curve_0101.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: catboost_training_curve_0101.png")

yp_final   = model_final.predict(X_test)
r2_final   = r2_score(y_test, yp_final)
mae_final  = mean_absolute_error(y_test, yp_final)
rmse_final = np.sqrt(mean_squared_error(y_test, yp_final))

print(f"\nCatBoost final:  R²={r2_final:.4f}  MAE={mae_final:.4f}  RMSE={rmse_final:.4f}")
print(f"Trees used: {model_final.best_iteration_}")

0:	learn: 0.3113986	test: 0.3662446	best: 0.3662446 (0)	total: 10.3ms	remaining: 10.3s
50:	learn: 0.1859756	test: 0.2557462	best: 0.2557462 (50)	total: 311ms	remaining: 5.79s
100:	learn: 0.1800995	test: 0.2492525	best: 0.2492525 (100)	total: 582ms	remaining: 5.18s
150:	learn: 0.1793291	test: 0.2482366	best: 0.2478860 (119)	total: 860ms	remaining: 4.84s
Stopped by overfitting detector  (40 iterations wait)

bestTest = 0.2478860498
bestIteration = 119

Shrink model to first 120 iterations.
Saved: catboost_training_curve_0101.png

CatBoost final:  R²=0.6712  MAE=0.1400  RMSE=0.1906
Trees used: 119


In [29]:
import numpy as np
from scipy.stats import pearsonr, spearmanr

# groupkfold

gkf = GroupKFold(n_splits=5)
fold_r2, fold_mae, fold_rmse = [], [], []
fold_pearson, fold_spearman = [], []  # Added tracking lists

print("\nGroupKFold evaluation (held-out broths):")

for fold_num, (tr_idx, te_idx) in enumerate(
    gkf.split(X, y, groups=groups), start=1
):

    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    held_broths = sorted(set(df.iloc[te_idx]["broth_id"]))
    held_divs = sorted(df.iloc[te_idx]["diversity_level"].unique())
    print(
        f"\nFold {fold_num}: broths={held_broths}  divs={held_divs}  n_test={len(X_te):,}"
    )

    # 1. Grab the broth groups belonging strictly to this fold's training set
    groups_tr = groups[tr_idx]

    # 2. Carve a localized, leak-free validation set out of these training broths
    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(
        X_tr, y_tr, groups_tr, frac=0.15, seed=42 + fold_num
    )

    # 3. Rebuild the pools dynamically for this specific fold
    fit_pool = Pool(Xg_fit, label=yg_fit, cat_features=[])
    val_pool = Pool(Xg_val, label=yg_val, cat_features=[])

    # 4. Initialize the model (varying the seed slightly per fold for stability)
    m_fold = CatBoostRegressor(
        **grid_search.best_params_,
        iterations=1000,
        subsample=0.8,
        early_stopping_rounds=40,
        use_best_model=True,
        eval_metric="RMSE",
        random_seed=42 + fold_num,
        thread_count=-1,
        verbose=0,
        train_dir="catboost_info_v1",
    )

    # 5. Fit using the newly carved local validation pool
    m_fold.fit(fit_pool, eval_set=val_pool)
    yp_fold = m_fold.predict(X_te)

    r2_f = r2_score(y_te, yp_fold)
    mae_f = mean_absolute_error(y_te, yp_fold)
    rmse_f = np.sqrt(mean_squared_error(y_te, yp_fold))

    # Calculate Pearson and Spearman correlations for this fold
    p_corr, _ = pearsonr(y_te, yp_fold)
    s_corr, _ = spearmanr(y_te, yp_fold)

    fold_r2.append(r2_f)
    fold_mae.append(mae_f)
    fold_rmse.append(rmse_f)
    fold_pearson.append(p_corr)  # Save metrics
    fold_spearman.append(s_corr)

    print(
        f"  R²={r2_f:.4f}  MAE={mae_f:.4f}  RMSE={rmse_f:.4f}  Pearson={p_corr:.4f}  Spearman={s_corr:.4f}"
    )

print(f"\nGroupKFold summary across {gkf.n_splits} folds:")
print(f"  R²:       mean={np.mean(fold_r2):.4f}  std={np.std(fold_r2):.4f}")
print(f"  MAE:      mean={np.mean(fold_mae):.4f}  std={np.std(fold_mae):.4f}")
print(f"  RMSE:     mean={np.mean(fold_rmse):.4f}  std={np.std(fold_rmse):.4f}")
print(
    f"  Pearson:  mean={np.mean(fold_pearson):.4f}  std={np.std(fold_pearson):.4f}"
)
print(
    f"  Spearman: mean={np.mean(fold_spearman):.4f}  std={np.std(fold_spearman):.4f}"
)

print(f"\nCompare to stratified-split R²={r2_final:.4f}")
print(
    f"Drop: {r2_final - np.mean(fold_r2):.4f}  (how much was broth memorisation)"
)


GroupKFold evaluation (held-out broths):

Fold 1: broths=['Broth_005', 'Broth_010', 'Broth_015', 'Broth_020', 'Broth_025', 'Broth_030', 'Broth_035', 'Broth_040', 'Broth_045', 'Broth_050', 'Broth_055', 'Broth_060', 'Broth_065', 'Broth_070', 'Broth_075', 'Broth_080', 'Broth_085', 'Broth_090', 'Broth_095', 'Broth_100', 'Broth_105', 'Broth_110', 'Broth_115', 'Broth_120', 'Broth_125', 'Broth_130', 'Broth_135', 'Broth_140', 'Broth_145', 'Broth_150']  divs=[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16)]  n_test=15,000
  R²=0.5773  MAE=0.1648  RMSE=0.2137  Pearson=0.7682  Spearman=0.7492

Fold 2: broths=['Broth_004', 'Broth_009', 'Broth_014', 'Broth_019', 'Broth_024', 'Broth_029', 'Broth_034', 'Broth_039', 'Broth_044', 'Broth_049', 'Broth_054', 'Broth_059', 'Broth_064', 'Broth_069', 'Broth_074', 'Broth_079', 'Broth_084', 'Broth_089', 'Broth_094', 'Broth_099', 'Broth_104', 'Broth_109', 'Broth_114', 'Broth_119', 'Broth_124', 'Broth_129', 'Broth_134', 'Broth_139', 'Broth_144',

In [17]:
gss = GroupShuffleSplit(n_splits=20, test_size=0.20, random_state=42)
r2_scores, mae_scores, rmse_scores = [], [], []

for split_num, (tr_idx, te_idx) in enumerate(gss.split(X, y, groups=groups), start=1):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

    groups_tr = groups[tr_idx]
    assert len(set(groups_tr).intersection(set(groups[te_idx]))) == 0

    Xg_fit, yg_fit, Xg_val, yg_val = inner_holdout(X_tr, y_tr, groups_tr, frac=0.15, seed=42 + split_num)
    fit_pool = Pool(Xg_fit, label=yg_fit, cat_features=[])
    val_pool = Pool(Xg_val, label=yg_val, cat_features=[])

    m = CatBoostRegressor(
        **grid_search.best_params_, iterations=1000, subsample=0.8,
        early_stopping_rounds=40, use_best_model=True, eval_metric='RMSE',
        random_seed=42 + split_num, thread_count=-1, verbose=0, train_dir='catboost_info_v3')
    m.fit(fit_pool, eval_set=val_pool)

    yp = m.predict(X_te)
    r2_scores.append(r2_score(y_te, yp))
    mae_scores.append(mean_absolute_error(y_te, yp))
    rmse_scores.append(np.sqrt(mean_squared_error(y_te, yp)))
    print(f"Split {split_num:02d}: R2={r2_scores[-1]:.4f}  MAE={mae_scores[-1]:.4f}  RMSE={rmse_scores[-1]:.4f}")

print("\nGroupShuffleSplit held-out-broth summary:")
print(f"  R2:   mean={np.mean(r2_scores):.4f}  std={np.std(r2_scores):.4f}")
print(f"  MAE:  mean={np.mean(mae_scores):.4f}  std={np.std(mae_scores):.4f}")
print(f"  RMSE: mean={np.mean(rmse_scores):.4f}  std={np.std(rmse_scores):.4f}")

Split 01: R2=0.0725  MAE=0.2653  RMSE=0.3369
Split 02: R2=0.5465  MAE=0.1767  RMSE=0.2276
Split 03: R2=0.3160  MAE=0.2176  RMSE=0.2636
Split 04: R2=0.4227  MAE=0.1850  RMSE=0.2427
Split 05: R2=0.5013  MAE=0.1751  RMSE=0.2186
Split 06: R2=0.4154  MAE=0.2057  RMSE=0.2437
Split 07: R2=0.3945  MAE=0.2238  RMSE=0.2548
Split 08: R2=0.3474  MAE=0.2067  RMSE=0.2638
Split 09: R2=0.4286  MAE=0.1997  RMSE=0.2436
Split 10: R2=0.3760  MAE=0.2006  RMSE=0.2611
Split 11: R2=0.3902  MAE=0.1863  RMSE=0.2474
Split 12: R2=0.3565  MAE=0.2300  RMSE=0.2676
Split 13: R2=0.3900  MAE=0.2192  RMSE=0.2589
Split 14: R2=0.1980  MAE=0.2174  RMSE=0.2910
Split 15: R2=0.4922  MAE=0.1851  RMSE=0.2416
Split 16: R2=0.6368  MAE=0.1784  RMSE=0.2095
Split 17: R2=0.2485  MAE=0.2151  RMSE=0.2975
Split 18: R2=0.4431  MAE=0.1964  RMSE=0.2366
Split 19: R2=0.3265  MAE=0.2322  RMSE=0.2791
Split 20: R2=0.3552  MAE=0.2005  RMSE=0.2533

GroupShuffleSplit held-out-broth summary:
  R2:   mean=0.3829  std=0.1193
  MAE:  mean=0.2058  std=

In [30]:
print("Performance by diversity level and community size")

results_df = pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'diversity_level': df.loc[X_test.index, 'diversity_level'].values,
    'community_size':  df.loc[X_test.index, 'community_size'].values,
})

def condition_metrics(y_true_arr, y_pred_arr):
    y_t      = np.asarray(y_true_arr)
    y_p      = np.asarray(y_pred_arr)
    r2       = r2_score(y_t, y_p)
    mae      = mean_absolute_error(y_t, y_p)
    rmse     = np.sqrt(mean_squared_error(y_t, y_p))
    pr, _    = pearsonr(y_t, y_p)
    sr, _    = spearmanr(y_t, y_p)
    bias     = float(np.mean(y_p - y_t))
    std_true = float(np.std(y_t))
    return {'n': len(y_t), 'r2': r2, 'mae': mae, 'rmse': rmse,
            'pearson_r': pr, 'spearman_r': sr,
            'bias': bias, 'std_true': std_true}

hdr = f"{'':6} {'R²':>7} {'MAE':>7} {'RMSE':>7} {'Pearson':>8} {'Spearman':>9} {'Bias':>8} {'std(y)':>7} {'N':>6}"
print("\nBy diversity level:")
print(hdr)
print("-" * len(hdr))
for d, grp in results_df.groupby('diversity_level'):
    m    = condition_metrics(grp['y_true'], grp['y_pred'])
    flag = "  *" if m['std_true'] < 0.08 else "   "
    print(f"D={d:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}{flag}")
print("  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.")

print("\nBy community size:")
print(hdr)
print("-" * len(hdr))
for n, grp in results_df.groupby('community_size'):
    m = condition_metrics(grp['y_true'], grp['y_pred'])
    print(f"N={n:<3}  {m['r2']:>7.4f} {m['mae']:>7.4f} {m['rmse']:>7.4f} "
          f"{m['pearson_r']:>8.4f} {m['spearman_r']:>9.4f} "
          f"{m['bias']:>8.4f} {m['std_true']:>7.4f} {m['n']:>6,}")

Performance by diversity level and community size

By diversity level:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
D=1     0.8489  0.1008  0.1683   0.9217    0.9375   0.0043  0.4330  3,000   
D=2     0.5741  0.1655  0.2326   0.7606    0.7889   0.0187  0.3564  3,000   
D=4     0.4309  0.1796  0.2221   0.6617    0.6706  -0.0181  0.2944  3,000   
D=8     0.5510  0.1407  0.1742   0.7428    0.7104   0.0003  0.2600  3,000   
D=16    0.3419  0.1132  0.1397   0.5912    0.5595  -0.0055  0.1722  3,000   
  * std(y) < 0.08: R² is unreliable here. Read MAE and Spearman r instead.

By community size:
            R²     MAE    RMSE  Pearson  Spearman     Bias  std(y)      N
-------------------------------------------------------------------------
N=12    0.4672  0.1751  0.2390   0.7870    0.7608   0.1277  0.3274  3,000
N=16    0.6368  0.1421  0.1994   0.8254    0.8106   0.0694  0.3309  3,000
N=24 

In [31]:
# shap analysis

print('shap analysis')

# CatBoost has its own built-in SHAP implementation accessible via
# get_feature_importance(type='ShapValues'), which is fast and exact.
# However, shap.TreeExplainer also supports CatBoost 
# we use it here to keep the interface identical across all three
# model scripts so results are directly comparable.
explainer   = shap.TreeExplainer(model_final)
rng         = np.random.default_rng(42)
idx         = rng.choice(len(X_test), size=2000, replace=False)
shap_values = explainer.shap_values(X_test.iloc[idx])
mean_shap   = pd.Series(np.abs(shap_values).mean(0), index=feature_cols).sort_values(ascending=False)

print(f"\nTop 15 features by mean |SHAP|:")
print(mean_shap.head(15).round(5).to_string())

shap analysis

Top 15 features by mean |SHAP|:
R0_c8         0.04309
R0_c6         0.03645
active_c8     0.03004
R0_c9         0.02693
R0_c20        0.02333
R0_c1         0.02286
R0_c16        0.02091
R0_c17        0.01677
active_c14    0.01546
R0_c13        0.01528
R0_c3         0.01406
R0_c10        0.01343
R0_c19        0.01272
active_c20    0.01253
R0_c2         0.01173


In [32]:
# figures

from matplotlib.patches import Patch

output_name = f"catboost_v{VERSION}_results_01.png"

fig = plt.figure(figsize=(18, 14), dpi=120)
fig.suptitle(
    f'CatBoost — {version_label}\n'
    f'R²={r2_final:.3f}  MAE={mae_final:.3f}  RMSE={rmse_final:.3f}',
    fontsize=12, fontweight='bold', y=0.99
)

ax1 = fig.add_subplot(2, 3, 1)
for d, col in zip([1,2,4,8,16], ['#e41a1c','#ff7f00','#4daf4a','#377eb8','#984ea3']):
    mask = results_df['diversity_level'].values == d
    ax1.scatter(y_test.values[mask], yp_final[mask], alpha=0.15, s=2, c=col, label=f'D={d}')
ax1.plot([0,1],[0,1],'r--',lw=1.5)
ax1.set_xlabel('True suppression'); ax1.set_ylabel('Predicted')
ax1.set_title(f'Predicted vs Actual  R²={r2_final:.3f}')
ax1.legend(fontsize=7, markerscale=3)

ax2 = fig.add_subplot(2, 3, 2)
resid = y_test.values - yp_final
for n, col in zip([12,16,24,32,48], ['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']):
    mask = results_df['community_size'].values == n
    ax2.scatter(yp_final[mask], resid[mask], alpha=0.15, s=2, c=col, label=f'N={n}')
ax2.axhline(0, color='r', linestyle='--', lw=1.5)
p5, p95 = np.percentile(resid, [5, 95])
ax2.axhline(p5,  color='grey', linestyle=':', lw=1, label=f'5th: {p5:.3f}')
ax2.axhline(p95, color='grey', linestyle=':', lw=1, label=f'95th: {p95:.3f}')
ax2.set_xlabel('Predicted'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals by Community Size')
ax2.legend(fontsize=7, markerscale=3)

ax3 = fig.add_subplot(2, 3, 3)
top_n    = min(25, len(feature_cols))
top_shap = mean_shap.head(top_n)
def feat_col(f):
    if 'Species' in f: return '#2196F3'
    if 'R0'      in f: return '#4CAF50'
    if 'active'  in f: return '#FF9800'
    return '#9C27B0'
cols = [feat_col(f) for f in top_shap.index]
ax3.barh(range(top_n), top_shap.values[::-1], color=cols[::-1])
ax3.set_yticks(range(top_n))
ax3.set_yticklabels(top_shap.index[::-1], fontsize=7)
ax3.set_xlabel('Mean |SHAP|'); ax3.set_title(f'Top {top_n} Features (SHAP)')
ax3.legend(handles=[
    Patch(fc='#2196F3', label='Species'),
    Patch(fc='#4CAF50', label='R0'),
    Patch(fc='#FF9800', label='Active'),
    Patch(fc='#9C27B0', label='Structural'),
], loc='lower right', fontsize=7)

ax4 = fig.add_subplot(2, 3, 4)
divs   = sorted(results_df['diversity_level'].unique())
r2_div = [r2_score(results_df[results_df['diversity_level']==d]['y_true'],
                   results_df[results_df['diversity_level']==d]['y_pred']) for d in divs]
ax4.bar([str(d) for d in divs], r2_div, color='steelblue', edgecolor='navy')
ax4.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax4.set_xlabel('Diversity level'); ax4.set_ylabel('R²')
ax4.set_title('R² by Diversity Level'); ax4.set_ylim(0, 1.05); ax4.legend(fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
sizes  = sorted(results_df['community_size'].unique())
r2_sz  = [r2_score(results_df[results_df['community_size']==n]['y_true'],
                   results_df[results_df['community_size']==n]['y_pred']) for n in sizes]
ax5.bar([str(n) for n in sizes], r2_sz, color='darkorange', edgecolor='saddlebrown')
ax5.axhline(r2_final, color='red', linestyle='--', lw=1.5, label=f'Overall R²={r2_final:.3f}')
ax5.set_xlabel('Community size N'); ax5.set_ylabel('R²')
ax5.set_title('R² by Community Size'); ax5.set_ylim(0, 1.05); ax5.legend(fontsize=8)

ax6 = fig.add_subplot(2, 3, 6)
groups, values, colours = [], [], []
sp_s = mean_shap[[f for f in feature_cols if 'Species' in f]].sum()
if sp_s > 0:
    groups.append('Species'); values.append(sp_s); colours.append('#2196F3')
r0_s = mean_shap[r0_cols].sum()
groups.append('R0'); values.append(r0_s); colours.append('#4CAF50')
ac_s = mean_shap[active_cols].sum()
groups.append('Active'); values.append(ac_s); colours.append('#FF9800')
st_s = mean_shap[[f for f in feature_cols if f in ['community_size','diversity_level']]].sum()
if st_s > 0:
    groups.append('Structural'); values.append(st_s); colours.append('#9C27B0')
ax6.pie(values, labels=groups, colors=colours,
        autopct='%.1f%%', startangle=90, textprops={'fontsize': 10})
ax6.set_title('SHAP by Feature Group')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(output_name, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nPlot saved: {output_name}")


Plot saved: catboost_v3_results_01.png


In [33]:
# save outputs 

model_save = f"catboost_v{VERSION}_model.cbm"
pred_save  = f"catboost_v{VERSION}_predictions.csv"

# CatBoost saves in its own .cbm (CatBoost Model) binary format.
# Reload with: m = CatBoostRegressor(); m.load_model('catboost_v1_model.cbm')
model_final.save_model(model_save)

pd.DataFrame({
    'y_true':          y_test.values,
    'y_pred':          yp_final,
    'residual':        y_test.values - yp_final,
    'diversity_level': results_df['diversity_level'].values,
    'community_size':  results_df['community_size'].values,
}).to_csv(pred_save, index=False)

print(f"Model saved:       {model_save}")
print(f"Predictions saved: {pred_save}")

Model saved:       catboost_v3_model.cbm
Predictions saved: catboost_v3_predictions.csv


In [34]:
# summary

print(f"\n{'='*60}")
print(f"SUMMARY — CatBoost VERSION {VERSION}: {version_label}")
print(f"{'='*60}")
print(f"Features:  {len(feature_cols)}")
print(f"R²:        {r2_final:.4f}")
print(f"MAE:       {mae_final:.4f}")
print(f"RMSE:      {rmse_final:.4f}")
print(f"Trees:     {model_final.best_iteration_}")
print(f"\nTop 5 features:")
for feat, val in mean_shap.head(5).items():
    print(f"  {feat:<25} {val:.5f}")


SUMMARY — CatBoost VERSION 3: Pure nutrients only (no species, no size)
Features:  40
R²:        0.6712
MAE:       0.1400
RMSE:      0.1906
Trees:     119

Top 5 features:
  R0_c8                     0.04309
  R0_c6                     0.03645
  active_c8                 0.03004
  R0_c9                     0.02693
  R0_c20                    0.02333
